## This is the main code of my system

Everything I need to run the live system is set up across the cells below. 
I run the cells from 2 onwards once, in order. this loads all the trained 
models, baselines, and helper functions I need. Once that's done, I can go 
back up to the live pipeline cell (moved to the top for easy access) and 
run it as many times as I want during a session, without needing to redo 
the setup each time — as long as I don't restart the kernel.

If I do restart the kernel, I have to run cells 2 onwards again first, 
before the live pipeline cell will work.

In [ ]:
#importing libraries 
import sounddevice as sd
from collections import deque
import numpy as np
from pythonosc import udp_client
import soundfile as sf
from pathlib import Path
from datetime import datetime


#coonecting with the interface
sd.default.device = [0, None]
#connecting with Processing 
osc_client = udp_client.SimpleUDPClient("127.0.0.1", 9000)


#window size of sound 
sr = 44100
block_size = 512
window_size = 20
audio_history_size = 4096
phrase_audio_size = int(sr * 2) 
ioi_window = 5

#rolling buffers for each feature, and for raw audio history
rms_buffer = deque(maxlen=window_size)
centroid_buffer = deque(maxlen=window_size)
onset_buffer = deque(maxlen=window_size)
audio_history = deque(maxlen=audio_history_size)
phrase_audio_history = deque(maxlen=phrase_audio_size)

#recording the sessions in order to be able to further compute the baselines after
sessions_folder = Path("daria-recordings/sessions")
sessions_folder.mkdir(exist_ok=True, parents=True)
session_filename = sessions_folder / f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.wav"
session_file = sf.SoundFile(str(session_filename), mode='w', samplerate=sr, channels=1)
print("Recording to:", session_filename)


#main function 
def callback(indata, frames, time, status):
    y = indata[:, 0]
    session_file.write(y)
    audio_history.extend(y)
    phrase_audio_history.extend(y)

    #RMS and centroid can be calculated directly from each rolling chunk 
    rms_val = librosa.feature.rms(y=y, frame_length=block_size, hop_length=block_size)[0][0]
    centroid_val = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=block_size, hop_length=block_size)[0][0]

    #onset needs a longer rolling history of audio before it can be calculated
    if len(audio_history) == audio_history_size:
        y_history = np.array(audio_history)
        onset_env = librosa.onset.onset_strength(y=y_history, sr=sr, hop_length=block_size)
        onset_val = onset_env[-1] if len(onset_env) > 0 else 0.0
    else:
        onset_val = 0.0

    #phrase continuity needs an even longer window, since it depends on detecting several onsets
    phrase_deviation = 0.0
    if len(phrase_audio_history) == phrase_audio_size:
        y_phrase = np.array(phrase_audio_history)
        onset_times_local = librosa.onset.onset_detect(y=y_phrase, sr=sr, hop_length=block_size, units='time')
        print("num onsets found:", len(onset_times_local), "last 3 onset times:", onset_times_local[-3:] if len(onset_times_local) >= 3 else onset_times_local)
        if len(onset_times_local) >= ioi_window + 1:
            recent_iois = np.diff(onset_times_local[-(ioi_window+1):])
            current_std = np.std(recent_iois)
            phrase_deviation = abs(current_std - baseline_std_ioi)
            print("current_std:", current_std, "phrase_deviation:", phrase_deviation)
    
    rms_buffer.append(rms_val)
    centroid_buffer.append(centroid_val)
    onset_buffer.append(onset_val)

    #once the buffer has 20 readings, compress each feature through its trained autoencoder
    #and compare it to the musician's trained baseline
    if len(rms_buffer) == window_size:
        result = encode_moment(np.array(rms_buffer), np.array(centroid_buffer), np.array(onset_buffer))
        rms_deviation = np.linalg.norm(result["rms"].flatten() - baseline_rms)
        centroid_deviation = np.linalg.norm(result["centroid"].flatten() - baseline_centroid)
        onset_deviation = np.linalg.norm(result["onset"].flatten() - baseline_onset)
        rms_raw_representative = max(rms_buffer)
        osc_client.send_message("/centroid", float(centroid_deviation))
        osc_client.send_message("/onset", float(onset_deviation))
        osc_client.send_message("/phrase", float(phrase_deviation))
        osc_client.send_message("/rms", float(rms_deviation))
        #RMS uses the raw value (not deviation) for the visualisation, reasoning is in the thesis
        osc_client.send_message("/rms_raw", float(rms_raw_representative))
        osc_client.send_message("/centroid_raw", float(centroid_val))
try:
    with sd.InputStream(samplerate=sr, blocksize=block_size, channels=1, device=0, callback=callback):
        print("Listening...")
        sd.sleep(120000)
except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    session_file.close()
    print("Session saved:", session_filename)

Recording to: daria-recordings/sessions/session_20260907_160206.wav
Listening...
num onsets found: 18 last 3 onset times: [1.79954649 1.92725624 1.9969161 ]
current_std: 0.04344055741560672 phrase_deviation: 0.08834778610523807
num onsets found: 18 last 3 onset times: [1.78793651 1.91564626 1.98530612]
current_std: 0.04344055741560672 phrase_deviation: 0.08834778610523807
num onsets found: 14 last 3 onset times: [1.77632653 1.90403628 1.97369615]
current_std: 0.06081678038184948 phrase_deviation: 0.0709715631389953
num onsets found: 19 last 3 onset times: [1.76471655 1.8924263  1.96208617]
current_std: 0.04344055741560665 phrase_deviation: 0.08834778610523814
num onsets found: 19 last 3 onset times: [1.88081633 1.95047619 1.9969161 ]
current_std: 0.03332690452760334 phrase_deviation: 0.09846143899324145
num onsets found: 14 last 3 onset times: [1.86920635 1.93886621 1.98530612]
current_std: 0.06833139911488274 phrase_deviation: 0.06345694440596204
num onsets found: 19 last 3 onset time

In [1]:
#import everything 
import torch
import librosa
import numpy as np
import sounddevice
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path
import torch.nn as nn
from pythonosc import udp_client

Load trained autoencoders

In [ ]:
#autoencoder class definition
class WindowAutoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=2, hidden_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model_rms = WindowAutoencoder()
model_rms.load_state_dict(torch.load("models/rms_autoencoder.pt"))
model_rms.eval()

model_centroid = WindowAutoencoder()
model_centroid.load_state_dict(torch.load("models/centroid_autoencoder.pt"))
model_centroid.eval()

model_onset = WindowAutoencoder()
model_onset.load_state_dict(torch.load("models/onset_autoencoder.pt"))
model_onset.eval()

WindowAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=20, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=2, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=20, bias=True)
  )
)

Feature extraction utilities 

In [3]:
#extract windows
def extract_windows(feature_array, window_size=20, step=1):
    windows = []
    for i in range(0, len(feature_array) - window_size, step):
        windows.append(feature_array[i:i + window_size])
    return np.array(windows)

def extract_all_features(clip_path):
    y, sr = librosa.load(clip_path, sr=44100, mono=True)
    rms = librosa.feature.rms(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    return extract_windows(rms), extract_windows(centroid), extract_windows(onset_env)

def encode_clip(rms_w, centroid_w, onset_w):
    def avg_latent(windows, model):
        x = torch.tensor(windows, dtype=torch.float32)
        with torch.no_grad():
            _, z = model(x)
        return z.numpy().mean(axis=0)
    return {
        "RMS": avg_latent(rms_w, model_rms),
        "Centroid": avg_latent(centroid_w, model_centroid),
        "Onset": avg_latent(onset_w, model_onset)
    }

Combine features in one encoded representation 


In [4]:
def encode_moment(rms_window, centroid_window, onset_window):
    with torch.no_grad():
        _, z_rms = model_rms(torch.tensor(rms_window, dtype=torch.float32).unsqueeze(0))
        _, z_centroid = model_centroid(torch.tensor(centroid_window, dtype=torch.float32).unsqueeze(0))
        _, z_onset = model_onset(torch.tensor(onset_window, dtype=torch.float32).unsqueeze(0))
        
        #audio_tensor = torch.from_numpy(audio_chunk).float().unsqueeze(0).unsqueeze(0)
        #z_rave = rave_model.encode(audio_tensor)
    
    return {
        "rms": z_rms.numpy(),
        "centroid": z_centroid.numpy(),
        "onset": z_onset.numpy(),
        #"rave": z_rave.numpy()
    }

Compute personal baseline from saved features

Uses the .npy feature files extracted from my own recordings, run through 
each trained autoencoder, then averaged to get one baseline position per 
feature. This baseline represents my typical playing, based on my 17 
training recordings.

In [ ]:
rms_scaled = np.load("RMS/RMS-features.npy")
centroid_scaled = np.load("Centroid/centroid-features.npy")
onset_scaled = np.load("ON-set/Onset-features.npy")

#run each feature through its trained autoencoder then average across all windows
# to get one single baseline position per feature
baseline_rms = model_rms.encoder(torch.tensor(rms_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
baseline_centroid = model_centroid.encoder(torch.tensor(centroid_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
baseline_onset = model_onset.encoder(torch.tensor(onset_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
#phrase continuity baseline is different - just the average IOI standard deviation, no autoencoder
baseline_std_ioi = float(np.load("onset-baseline/baseline_std_ioi.npy"))


print(baseline_rms, baseline_centroid, baseline_onset, baseline_std_ioi)

[-1.1723264  -0.59157836] [-1.1591513  0.1669771] [ 0.24743259 -0.29686362] 0.1317883435208448


Offline version of the live pipeline - runs a whole saved file through the same eature extraction + autoencoder process, instead of live audio chunks

In [9]:
def encode_full_clip(clip_path, window_size=20):
    y, sr = librosa.load(clip_path, sr=44100, mono=True)
    
    rms = librosa.feature.rms(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    
    rms_windows = extract_windows(rms, window_size)
    centroid_windows = extract_windows(centroid, window_size)
    onset_windows = extract_windows(onset_env, window_size)
    
    n_steps = min(len(rms_windows), len(centroid_windows), len(onset_windows))
    
    results = []
    for i in range(n_steps):
        result = encode_moment(rms_windows[i], centroid_windows[i], onset_windows[i])
        results.append(result)
    
    return results

Offline test: phrase continuity deviation across a full recording

In [13]:
y, sr = librosa.load("daria-recordings/chitara-pt-cod.wav", sr=44100, mono=True)
onset_times = librosa.onset.onset_detect(y=y, sr=sr, units='time')

window = 5
phrase_deviations = []

for i in range(window, len(onset_times) - window):
    recent_iois = np.diff(onset_times[i-window:i+1])
    current_std = np.std(recent_iois)
    deviation = abs(current_std - baseline_std_ioi)
    phrase_deviations.append(deviation)

print(min(phrase_deviations), max(phrase_deviations), np.mean(phrase_deviations))

0.04324751727097577 0.11536935613002806 0.07504200418848447


Test to find interface channel 

In [8]:
import sounddevice as sd
devices = sd.query_devices()
for i, d in enumerate(devices):
    print(i, d['name'], f"({d['max_input_channels']} in, {d['max_output_channels']} out)")

0 Scarlett 4i4 USB (6 in, 4 out)
1 MacBook Pro Microphone (1 in, 0 out)
2 MacBook Pro Speakers (0 in, 2 out)
3 iPhone (2) Microphone (1 in, 0 out)
4 Microsoft Teams Audio (1 in, 1 out)
5 ZoomAudioDevice (2 in, 2 out)


Debugging RMS 

In [1]:
import sounddevice as sd
import numpy as np
import librosa

recording = sd.rec(int(5 * 44100), samplerate=44100, channels=1, device=0)
sd.wait()

y = recording[:, 0]
rms_vals = librosa.feature.rms(y=y, frame_length=512, hop_length=512)[0]
print("Max RMS:", np.max(rms_vals))
print("Mean RMS:", np.mean(rms_vals))

Max RMS: 0.042174082
Mean RMS: 0.0029549852


Test for RMS values

In [3]:
import sounddevice as sd
import numpy as np

recording = sd.rec(int(5 * 44100), samplerate=44100, channels=1, device=0)
sd.wait()
print("Max amplitude:", np.max(np.abs(recording)))
print("Mean amplitude:", np.mean(np.abs(recording)))



Max amplitude: 6.2879133
Mean amplitude: 0.005678557
